**Step 1 & 2: Text data collection and preprocessing (Data Cleaning)**

In [7]:
import string
import os

def load_and_clean_captions(file_path):
    captions_dict = {}
    table = str.maketrans('', '', string.punctuation)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            parts = line.split('\t') if '\t' in line else line.split(',', 1)
            
            if parts[0] == 'image' or len(parts) < 2:
                continue
                
            image_id, caption = parts[0], parts[1]
            
            image_id = image_id.split('#')[0].replace('.jpg', '')
            
            caption = caption.lower() 
            caption = caption.translate(table)
            
            words = [word for word in caption.split() if word.isalpha() and len(word) > 1]
            
            if image_id not in captions_dict:
                captions_dict[image_id] = []
            captions_dict[image_id].append(" ".join(words))
            
    return captions_dict

dataset_path = 'dataset/captions.txt' 
captions_mapping = load_and_clean_captions(dataset_path)

print(f"تعداد کل تصاویر پردازش شده: {len(captions_mapping)}")
print("-" * 30)

first_key = list(captions_mapping.keys())[0]
print(f"Image ID: {first_key}")
print(f"Captions: {captions_mapping[first_key]}")

تعداد کل تصاویر پردازش شده: 8091
------------------------------
Image ID: 1000268201_693b08cb0e
Captions: ['child in pink dress is climbing up set of stairs in an entry way', 'girl going into wooden building', 'little girl climbing into wooden playhouse', 'little girl climbing the stairs to her playhouse', 'little girl in pink dress going into wooden cabin']


**Step 3: Vocabulary Building and Word Filtering**

In [8]:
from collections import Counter

def build_vocabulary(captions_mapping, min_count=10):
    all_words = []
    for caps in captions_mapping.values():
        for cap in caps:
            all_words.extend(cap.split())
    
    word_counts = Counter(all_words)
    
    vocabulary = [word for word, count in word_counts.items() if count >= min_count]
    
    print(f"تعداد کل کلمات یکتا (قبل از فیلتر): {len(word_counts)}")
    print(f"تعداد کل کلمات لغت‌نامه نهایی (تکرار >= {min_count}): {len(vocabulary)}")
    
    return set(vocabulary)

vocab = build_vocabulary(captions_mapping, min_count=10)

تعداد کل کلمات یکتا (قبل از فیلتر): 8763
تعداد کل کلمات لغت‌نامه نهایی (تکرار >= 10): 1947


In [9]:
import random

all_img_ids = list(captions_mapping.keys())

random.seed(42) 
random.shuffle(all_img_ids)

split_index = int(len(all_img_ids) * 0.8)

train_ids = all_img_ids[:split_index]
test_ids = all_img_ids[split_index:]

train_captions = {}
for img_id in train_ids:
    if img_id in captions_mapping:
        train_captions[img_id] = []
        for cap in captions_mapping[img_id]:
            labeled_cap = 'startseq ' + cap + ' endseq'
            train_captions[img_id].append(labeled_cap)

print(f"تعداد کل تصاویر: {len(all_img_ids)}")
print(f"تعداد تصاویر آموزشی: {len(train_ids)}")
print(f"تعداد تصاویر تست: {len(test_ids)}")

sample_id = train_ids[0]
print(f"\nنمونه کپشن آموزشی برای {sample_id}:")
print(train_captions[sample_id][0])


تعداد کل تصاویر: 8091
تعداد تصاویر آموزشی: 6472
تعداد تصاویر تست: 1619

نمونه کپشن آموزشی برای 2874984466_1aafec2c9f:
startseq black and white dog is playing with sheep in field endseq


ث

In [13]:
import os
import pickle
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

all_img_ids = list(captions_mapping.keys()) 

project_dir = os.getcwd() 
images_dir = os.path.join(project_dir, 'dataset', 'Images')
weights_path = os.path.join(project_dir, 'models', 'inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5')
features_save_path = os.path.join(project_dir, 'features.pkl')

if not os.path.exists(weights_path):
    print(f" خطا: فایل وزن در این مسیر پیدا نشد: {weights_path}")
else:
    print(" در حال بارگذاری مدل InceptionV3...")
    base_model = InceptionV3(weights=weights_path, include_top=False, pooling='avg')
    print(" مدل با موفقیت بارگذاری شد.")

def extract_features(directory, img_ids):
    features = {}
    print(f" شروع استخراج ویژگی برای {len(img_ids)} تصویر...")
    
    for i, img_id in enumerate(img_ids):
        filename = os.path.join(directory, img_id + '.jpg')
        
        if not os.path.exists(filename): 
            print(f" هشدار: تصویر {filename} یافت نشد.")
            continue
            
        try:
            image = load_img(filename, target_size=(299, 299))
            image = img_to_array(image)
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            
            feature = base_model.predict(image, verbose=0)
            features[img_id] = feature.reshape(-1)
            
            if (i+1) % 500 == 0: 
                print(f" پیشرفت: {i+1}/{len(img_ids)}")
        except Exception as e:
            print(f" خطا در پردازش تصویر {img_id}: {e}")
            
    return features

if os.path.exists(features_save_path):
    print(" فایل features.pkl موجود است. در حال بارگذاری...")
    with open(features_save_path, "rb") as f: 
        features = pickle.load(f)
    print(f" ویژگی‌ها بارگذاری شدند. تعداد: {len(features)}")
else:
    print("فایل ویژگی‌ها موجود نیست. در حال استخراج از تصاویر")
    features = extract_features(images_dir, all_img_ids) 
    with open(features_save_path, "wb") as f: 
        pickle.dump(features, f)
    print(" استخراج تمام شد و فایل features.pkl ذخیره گردید.")


 در حال بارگذاری مدل InceptionV3...
 مدل با موفقیت بارگذاری شد.
 فایل features.pkl موجود است. در حال بارگذاری...
 ویژگی‌ها بارگذاری شدند. تعداد: 8091


ج


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np

def to_lines(descriptions):
    all_desc = []
    for key in descriptions.keys():
        [all_desc.append(d) for d in descriptions[key]]
    return all_desc

def create_tokenizer(descriptions):
    lines = to_lines(descriptions)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer

def max_length(descriptions):
    lines = to_lines(descriptions)
    return max(len(d.split()) for d in lines)

tokenizer = create_tokenizer(train_captions)
vocab_size = len(tokenizer.word_index) + 1  
max_len = max_length(train_captions)

print(f' اندازه لغت‌نامه (Vocab Size): {vocab_size}')
print(f' حداکثر طول کپشن (Max Length): {max_len}')

def data_generator(descriptions, photos, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0
    while True:
        for key, desc_list in descriptions.items():
            n += 1
            photo = photos[key][0] 
            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    
                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)
            
            if n == batch_size:
                yield [np.array(X1), np.array(X2)], np.array(y)
                X1, X2, y = list(), list(), list()
                n = 0

 اندازه لغت‌نامه (Vocab Size): 7934
 حداکثر طول کپشن (Max Length): 34


چ

In [15]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

def define_model(vocab_size, max_length):
    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    
    return model

model = define_model(vocab_size, max_len)

print(model.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)    │ (None, 34)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_5 (InputLayer)    │ (None, 2048)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 34, 256)           │       2,031,104 │ input_layer_6[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 2048)              │               0 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 34, 256)           │               0 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 34)                │               0 │ input_layer_6[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 256)               │         524,544 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 256)               │         525,312 │ dropout_1[0][0],           │
│                               │                           │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 256)               │               0 │ dense[0][0], lstm[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 256)               │          65,792 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 7934)              │       2,039,038 │ dense_1[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 5,185,790 (19.78 MB)

 Trainable params: 5,185,790 (19.78 MB)

 Non-trainable params: 0 (0.00 B)

None


ح